# 5. Remotes and origin

A **remote** is another copy of your repository, usually on a server like
GitHub. **origin** is the usual name for it.

Here we build our own "server" in a folder, so you can see there is no
magic. Three folders:

```mermaid
flowchart LR
    me["barge (you)"] -->|git push| origin["barge.git (the server)"]
    origin -->|git fetch| me
    origin -->|git clone| deniz["deniz (a colleague)"]
    deniz -->|git push| origin
```

In [ ]:
!rm -rf ~/git-course/05-remotes
!mkdir -p ~/git-course/05-remotes
%cd ~/git-course/05-remotes

## The server

A server repository is **bare**: it has the history but no working files,
because nobody edits files on a server.

In [ ]:
!git init --bare barge.git

## Your repository

In [ ]:
!mkdir barge
%cd barge

In [ ]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge."""

RHO = 1025.0  # kg/m^3, sea water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)

In [ ]:
!git init
!git add hydrostatics.py
!git commit -m "Add volume and displacement for a box barge"

## Adding a remote

`git remote add` gives the server a name. The name is `origin` by
convention. On GitHub the address would look like
`https://github.com/you/barge.git`; here it is just a folder.

In [ ]:
!git remote add origin ../barge.git

In [ ]:
!git remote -v

## Pushing

`git push` sends your commits to the server. `-u` is needed the first time
only: it links your `main` to the server's `main`.

In [ ]:
!git push -u origin main

In [ ]:
!git status

## A colleague clones it

`git clone` copies a repository from the server, including all its history.
The clone knows `origin` automatically.

In [ ]:
%cd ..
!git clone barge.git deniz

In [ ]:
%cd deniz
!git config user.name "Deniz"
!git log --oneline

Deniz adds the Froude number and pushes:

In [ ]:
%%writefile -a hydrostatics.py


def froude_number(speed, length):
    """Speed-length ratio that scales wave resistance."""
    return speed / (9.81 * length) ** 0.5

In [ ]:
!git add hydrostatics.py
!git commit -m "Add the Froude number"
!git push

## Fetch: what's new?

Back in our repository, git does not know about Deniz's commit yet — it
never talks to the server unless you ask.

In [ ]:
%cd ../barge
!git status

`git fetch` downloads new commits from the server, but does not change
your files:

In [ ]:
!git fetch

In [ ]:
!git status

In [ ]:
!git log --oneline --all

`origin/main` is where the server's `main` is. Ours is one commit behind.

## Pull: fetch and merge

`git pull` fetches and then merges in one step:

In [ ]:
!git pull

In [ ]:
!cat hydrostatics.py

## When both of you changed something

We add a function...

In [ ]:
%%writefile -a hydrostatics.py


def tonnes_per_cm(length, beam):
    """Tonnes of cargo that sink the barge by one centimetre."""
    return length * beam * RHO / 100000.0

In [ ]:
!git add hydrostatics.py
!git commit -m "Add tonnes per centimetre immersion"

...and at the same time Deniz changes the first line and pushes:

In [ ]:
%cd ../deniz

In [ ]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge, in SI units (m, kg, N)."""

RHO = 1025.0  # kg/m^3, sea water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)


def froude_number(speed, length):
    """Speed-length ratio that scales wave resistance."""
    return speed / (9.81 * length) ** 0.5

In [ ]:
!git add hydrostatics.py
!git commit -m "Say which units the module uses"
!git push

Now we try to push:

In [ ]:
%cd ../barge

In [ ]:
# This one fails on purpose: the server has a commit we don't have.
!git push

**Rejected.** The server will not throw away Deniz's commit to take ours.
We must get Deniz's work first. But a plain `git pull` refuses too, because
both sides have new commits and git wants us to choose how to join them:

In [ ]:
# This one fails on purpose: git asks us to choose.
!git pull

The two choices are chapters 3 and 4:

- `git pull --no-rebase` — join them with a merge commit.
- `git pull --rebase` — put our commit on top of Deniz's.

Our commit was never shared, so rebasing it is safe:

In [ ]:
!git pull --rebase

In [ ]:
!git push

In [ ]:
!git log --oneline --graph --all

## Pushing a branch

Branches stay on your computer until you push them:

In [ ]:
!git switch -c wetted-surface

In [ ]:
%%writefile -a hydrostatics.py


def wetted_surface(length, beam, draft):
    """Rough wetted surface area, in m^2."""
    return length * (beam + 2 * draft)

In [ ]:
!git add hydrostatics.py
!git commit -m "Add a rough wetted surface estimate"
!git push -u origin wetted-surface

In [ ]:
!git branch -a

On GitHub, this is when you would open a pull request (chapter 6).

## Summary

| I want to… | Command |
| - | - |
| copy a repository | `git clone <url>` |
| name a server | `git remote add origin <url>` |
| send my commits | `git push` |
| get new commits, without touching my files | `git fetch` |
| get new commits and merge them | `git pull` |
| get new commits, my work on top | `git pull --rebase` |
| push a new branch | `git push -u origin <branch>` |

Never use `git push --force` on a shared branch: it deletes other people's
commits from the server.

## Exercises

1. In `deniz`, run `git log --oneline --all` and compare with `barge`.
2. Make both repositories change the *same* line, then `git pull --rebase`
   and resolve the conflict.

Next: [6. Forks](06-forks.ipynb)